# `subvillage` 03: related-feature analysis

            **Purpose:** identify predictors that represent the same concept, form a
            hierarchy, share a missingness process or plausibly interact with
            `subvillage`.

            ## Relationships selected in advance

            - `ward` — Subvillages sit within wards, although names are reused.
- `lga` — LGA context partially disambiguates repeated subvillage names.
- `wpt_name` — Waterpoint names can repeat within local settlements.
- `region` — Region is the broad administrative back-off.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'subvillage'
feature_metadata = {'order': 11, 'name': 'subvillage', 'audit_type': 'high-cardinality-category', 'role': 'candidate', 'disposition': 'exclude raw one-hot value from the first baseline', 'finding': 'The field is extremely sparse, has reused names and exposes many test rows to unseen levels.', 'decision': 'Use only a separately validated hashing or frequency treatment with explicit missingness.', 'risk': 'Direct encoding encourages local memorisation and poor unseen coverage.', 'related': [{'feature': 'ward', 'reason': 'Subvillages sit within wards, although names are reused.'}, {'feature': 'lga', 'reason': 'LGA context partially disambiguates repeated subvillage names.'}, {'feature': 'wpt_name', 'reason': 'Waterpoint names can repeat within local settlements.'}, {'feature': 'region', 'reason': 'Region is the broad administrative back-off.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for subvillage.


In [2]:
relationship_inventory = pd.DataFrame(feature_metadata["related"])
display(relationship_inventory)

relationship_evidence = related_feature_summary(
    training_features,
    feature,
    feature_metadata["audit_type"],
    feature_metadata["related"],
    feature_types,
)
display(relationship_evidence)


,feature,reason
0,ward,"Subvillages sit within wards, although names a..."
1,lga,LGA context partially disambiguates repeated s...
2,wpt_name,Waterpoint names can repeat within local settl...
3,region,Region is the broad administrative back-off.


,primary,related,measure,association,complete rows,primary levels,related levels,forward modal purity (%),reverse modal purity (%),relationship rationale
0,subvillage,ward,bias-corrected Cramer's V,0.6128,59400,19288,2092,74.45,18.97,"Subvillages sit within wards, although names a..."
1,subvillage,lga,bias-corrected Cramer's V,0.6454,59400,19288,125,79.41,4.82,LGA context partially disambiguates repeated s...
2,subvillage,wpt_name,bias-corrected Cramer's V,0.3046,59400,19288,37398,41.37,68.23,Waterpoint names can repeat within local settl...
3,subvillage,region,bias-corrected Cramer's V,0.6751,59400,19288,21,83.43,2.74,Region is the broad administrative back-off.


## Discussion and modelling consequence

The relationships above were nominated before inspecting the pairwise
coefficients. A strong association can mean useful interaction, hierarchy,
shared collection behaviour or redundancy; it is not a reason to keep both
fields automatically.

For `subvillage`, carry the relationships into controlled ablations
and fit every learned grouping or encoding inside the training fold. The
current provisional disposition remains: **exclude raw one-hot value from the first baseline**.
